Disclaimer - This code is frankensteined together from several iterations and many file name/storage changes.

Much of it might not work if run right now, and I am kind of scared to touch it again anyway. It is here simply to give an overview of my process.

In [ ]:
# Code block to download everything except Facebook.

# %%time

# import requests
# import pandas as pd
# from pathlib import Path
# from datetime import date, timedelta
# from dateutil.relativedelta import relativedelta
# import time


# # -------------------------
# # SETTINGS
# # -------------------------

# API_KEY_PATH = "api_keys/massive.txt"

# YEARS = 5
# SAFETY_DAYS = 1
# RATE_LIMIT_SECONDS = 0.2

# EXCHANGES = ["XNYS", "XNAS"]


# # -------------------------
# # API HELPERS
# # -------------------------

# def load_api_key(filepath=API_KEY_PATH):
#     return Path(filepath).read_text(encoding="utf-8").strip()


# def get_all_pages(url, params=None):
#     all_results = []

#     api_key = params.get("apiKey") if params else None

#     while url:
#         response = requests.get(url, params=params, timeout=30)
#         response.raise_for_status()

#         data = response.json()

#         if data.get("status") == "NOT_AUTHORIZED":
#             raise PermissionError(
#                 data.get("message", "Not authorized for this request.")
#             )

#         all_results.extend(data.get("results", []))

#         url = data.get("next_url")

#         if url and api_key and "apiKey=" not in url:
#             separator = "&" if "?" in url else "?"
#             url = f"{url}{separator}apiKey={api_key}"

#         params = None

#     return all_results


# # -------------------------
# # TICKER UNIVERSE
# # -------------------------

# def get_massive_ticker_reference(api_key_path=API_KEY_PATH):

#     api_key = load_api_key(api_key_path)

#     url = "https://api.massive.com/v3/reference/tickers"

#     params = {
#         "market": "stocks",
#         "active": "true",
#         "sort": "ticker",
#         "limit": 1000,
#         "apiKey": api_key,
#     }

#     results = get_all_pages(url, params)

#     if not results:
#         raise ValueError("No ticker reference data returned.")

#     return pd.DataFrame(results)


# # -------------------------
# # COMPANY INFO
# # -------------------------

# def get_ticker_details(ticker, api_key_path=API_KEY_PATH):

#     api_key = load_api_key(api_key_path)

#     url = f"https://api.massive.com/v3/reference/tickers/{ticker}"

#     params = {
#         "apiKey": api_key
#     }

#     response = requests.get(
#         url,
#         params=params,
#         timeout=30
#     )

#     response.raise_for_status()

#     data = response.json()

#     result = data.get("results", {})

#     return {
#         "ticker": ticker,
#         "name": result.get("name"),
#         "market_cap": result.get("market_cap"),
#         "exchange": result.get("primary_exchange"),
#         "country": result.get("locale"),
#     }


# # -------------------------
# # DAILY BAR FETCHING
# # -------------------------

# def get_massive_daily_bars(
#     ticker,
#     api_key_path=API_KEY_PATH,
#     years=YEARS,
#     safety_days=SAFETY_DAYS,
#     adjusted=True,
#     sort="asc"
# ):

#     api_key = load_api_key(api_key_path)

#     today = date.today()

#     start_date = (
#         today
#         - relativedelta(years=years)
#         + timedelta(days=safety_days)
#     )

#     url = (
#         f"https://api.massive.com/v2/aggs/ticker/"
#         f"{ticker}/range/1/day/"
#         f"{start_date.isoformat()}/{today.isoformat()}"
#     )

#     params = {
#         "adjusted": str(adjusted).lower(),
#         "sort": sort,
#         "limit": 50000,
#         "apiKey": api_key,
#     }

#     results = get_all_pages(url, params)

#     if not results:
#         raise ValueError(f"No data returned for {ticker}.")

#     df = pd.DataFrame(results).rename(columns={
#         "t": "timestamp",
#         "o": "open",
#         "h": "high",
#         "l": "low",
#         "c": "close",
#         "v": "volume",
#         "vw": "vwap",
#         "n": "transactions",
#     })

#     df["timestamp"] = pd.to_datetime(
#         df["timestamp"],
#         unit="ms"
#     ).dt.date

#     df["ticker"] = ticker

#     preferred_order = [
#         "timestamp",
#         "ticker",
#         "open",
#         "high",
#         "low",
#         "close",
#         "volume",
#         "vwap",
#         "transactions",
#     ]

#     return df[
#         [col for col in preferred_order if col in df.columns]
#     ]


# # -------------------------
# # GET TICKER UNIVERSE
# # -------------------------

# df_all_tickers = get_massive_ticker_reference()

# filtered = df_all_tickers[
#     (df_all_tickers["primary_exchange"].isin(EXCHANGES)) &
#     (df_all_tickers["type"] == "CS") &
#     (df_all_tickers["active"] == True)
# ].copy()

# filtered = (
#     filtered
#     .sort_values("ticker")
#     .reset_index(drop=True)
# )

# tickers = (
#     filtered["ticker"]
#     .dropna()
#     .unique()
#     .tolist()
# )

# print(f"Tickers selected: {len(tickers)}")


# # -------------------------
# # COMPANY INFO FETCHING
# # -------------------------

# company_info_list = []

# for ticker in tickers:

#     try:
#         info = get_ticker_details(ticker)

#         company_info_list.append(info)

#         time.sleep(RATE_LIMIT_SECONDS)

#     except Exception as e:

#         print(f"{ticker} details failed: {e}")


# df_company_info = pd.DataFrame(company_info_list)


# # -------------------------
# # DAILY BAR FETCHING
# # -------------------------

# data = {}
# failed = []

# for ticker in tickers:

#     try:
#         df = get_massive_daily_bars(ticker)

#         data[ticker] = df

#         time.sleep(RATE_LIMIT_SECONDS)

#     except Exception as e:

#         print(f"{ticker} failed: {e}")

#         failed.append({
#             "ticker": ticker,
#             "error": str(e)
#         })


# # -------------------------
# # COMBINED DATAFRAME
# # -------------------------

# if data:

#     combined_df = pd.concat(
#         data.values(),
#         ignore_index=True
#     )

#     combined_df = combined_df.merge(
#         df_company_info,
#         on="ticker",
#         how="left"
#     )

#     final_cols = [
#         "timestamp",
#         "ticker",
#         "open",
#         "high",
#         "low",
#         "close",
#         "volume",
#         "vwap",
#         "transactions",
#         "name",
#         "market_cap",
#         "exchange",
#         "country"
#     ]

#     combined_df = (
#         combined_df[final_cols]
#         .sort_values(["ticker", "timestamp"])
#         .reset_index(drop=True)
#     )

#     print(f"Combined dataframe shape: {combined_df.shape}")

# else:

#     combined_df = pd.DataFrame()

#     print("No data was successfully fetched.")


# # -------------------------
# # FAILED TICKERS
# # -------------------------

# failed_df = pd.DataFrame(failed)

# if not failed_df.empty:

#     print(f"Failed tickers: {len(failed_df)}")

In [ ]:
# all_safe_df = combined_df.copy()
# all_safe_df.to_csv('daily_data/daily_raw_except_fb.csv', index=False)

In [7]:
# Download FB

# %%time

# import requests
# import pandas as pd
# from pathlib import Path
# from datetime import date, timedelta
# from dateutil.relativedelta import relativedelta


# # -------------------------
# # SETTINGS
# # -------------------------

# API_KEY_PATH = "api_keys/massive.txt"

# YEARS = 5
# SAFETY_DAYS = 1


# # -------------------------
# # API HELPERS
# # -------------------------

# def load_api_key(filepath=API_KEY_PATH):
#     return Path(filepath).read_text(encoding="utf-8").strip()


# def get_all_pages(url, params=None):

#     all_results = []

#     api_key = params.get("apiKey") if params else None

#     while url:

#         response = requests.get(
#             url,
#             params=params,
#             timeout=30
#         )

#         response.raise_for_status()

#         data = response.json()

#         if data.get("status") == "NOT_AUTHORIZED":
#             raise PermissionError(
#                 data.get("message", "Not authorized for this request.")
#             )

#         all_results.extend(data.get("results", []))

#         url = data.get("next_url")

#         if url and api_key and "apiKey=" not in url:

#             separator = "&" if "?" in url else "?"

#             url = f"{url}{separator}apiKey={api_key}"

#         params = None

#     return all_results


# # -------------------------
# # COMPANY INFO
# # -------------------------

# def get_ticker_details(ticker, api_key_path=API_KEY_PATH):

#     api_key = load_api_key(api_key_path)

#     url = f"https://api.massive.com/v3/reference/tickers/{ticker}"

#     params = {
#         "apiKey": api_key
#     }

#     response = requests.get(
#         url,
#         params=params,
#         timeout=30
#     )

#     response.raise_for_status()

#     data = response.json()

#     result = data.get("results", {})

#     return {
#         "ticker": ticker,
#         "name": result.get("name"),
#         "market_cap": result.get("market_cap"),
#         "exchange": result.get("primary_exchange"),
#         "country": result.get("locale"),
#     }


# # -------------------------
# # DAILY BAR FETCHING
# # -------------------------

# def get_massive_daily_bars(
#     ticker,
#     api_key_path=API_KEY_PATH,
#     years=YEARS,
#     safety_days=SAFETY_DAYS,
#     adjusted=True,
#     sort="asc"
# ):

#     api_key = load_api_key(api_key_path)

#     today = date.today()

#     start_date = (
#         today
#         - relativedelta(years=years)
#         + timedelta(days=safety_days)
#     )

#     url = (
#         f"https://api.massive.com/v2/aggs/ticker/"
#         f"{ticker}/range/1/day/"
#         f"{start_date.isoformat()}/{today.isoformat()}"
#     )

#     params = {
#         "adjusted": str(adjusted).lower(),
#         "sort": sort,
#         "limit": 50000,
#         "apiKey": api_key,
#     }

#     results = get_all_pages(url, params)

#     if not results:
#         raise ValueError(f"No data returned for {ticker}.")

#     df = pd.DataFrame(results).rename(columns={
#         "t": "timestamp",
#         "o": "open",
#         "h": "high",
#         "l": "low",
#         "c": "close",
#         "v": "volume",
#         "vw": "vwap",
#         "n": "transactions",
#     })

#     df["timestamp"] = pd.to_datetime(
#         df["timestamp"],
#         unit="ms"
#     ).dt.date

#     df["ticker"] = ticker

#     preferred_order = [
#         "timestamp",
#         "ticker",
#         "open",
#         "high",
#         "low",
#         "close",
#         "volume",
#         "vwap",
#         "transactions",
#     ]

#     return df[
#         [col for col in preferred_order if col in df.columns]
#     ]


# # -------------------------
# # FB ONLY
# # -------------------------

# ticker = "FB"


# # -------------------------
# # COMPANY INFO
# # -------------------------

# company_info = pd.DataFrame([
#     get_ticker_details(ticker)
# ])


# # -------------------------
# # DAILY DATA
# # -------------------------

# daily_df = get_massive_daily_bars(ticker)


# # -------------------------
# # FINAL DATAFRAME
# # -------------------------

# combined_df = daily_df.merge(
#     company_info,
#     on="ticker",
#     how="left"
# )

# final_cols = [
#     "timestamp",
#     "ticker",
#     "open",
#     "high",
#     "low",
#     "close",
#     "volume",
#     "vwap",
#     "transactions",
#     "name",
#     "market_cap",
#     "exchange",
#     "country"
# ]

# combined_df = (
#     combined_df[final_cols]
#     .sort_values(["ticker", "timestamp"])
#     .reset_index(drop=True)
# )

# print(combined_df.shape)

(492, 13)
CPU times: user 33.8 ms, sys: 5.33 ms, total: 39.2 ms
Wall time: 730 ms


In [9]:
# fb_df = combined_df.copy()
# fb_df.to_csv('daily_data/daily_raw_fb.csv', index=False)

In [10]:
import pandas as pd
all_safe_df = pd.read_csv('daily_data/daily_raw_except_fb.csv')
fb_df = pd.read_csv('daily_data/daily_raw_fb.csv')

In [16]:
all_safe_df[all_safe_df['ticker'] == 'META']

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,name,market_cap,exchange,country
2843116,2021-06-30,META,15.0800,15.1900,15.0420,15.12,3.349350e+05,15.1107,2101.0,"Meta Platforms, Inc. Class A Common Stock",1.535898e+12,XNAS,us
2843117,2021-07-01,META,15.1300,15.1300,14.8400,14.89,2.416290e+05,15.0126,2092.0,"Meta Platforms, Inc. Class A Common Stock",1.535898e+12,XNAS,us
2843118,2021-07-02,META,15.0200,15.1000,14.9300,15.00,3.881520e+05,15.0224,4733.0,"Meta Platforms, Inc. Class A Common Stock",1.535898e+12,XNAS,us
2843119,2021-07-06,META,15.0800,15.0900,14.8700,15.01,6.850940e+05,15.1119,11269.0,"Meta Platforms, Inc. Class A Common Stock",1.535898e+12,XNAS,us
2843120,2021-07-07,META,15.0601,15.1500,14.8400,14.89,3.625380e+05,14.9830,5538.0,"Meta Platforms, Inc. Class A Common Stock",1.535898e+12,XNAS,us
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2844250,2026-05-15,META,613.9900,621.1999,609.3100,614.23,1.327257e+07,614.6824,309867.0,"Meta Platforms, Inc. Class A Common Stock",1.535898e+12,XNAS,us
2844251,2026-05-18,META,609.1050,615.5900,603.6900,611.21,1.377244e+07,609.9810,376501.0,"Meta Platforms, Inc. Class A Common Stock",1.535898e+12,XNAS,us
2844252,2026-05-19,META,608.9200,613.9300,600.5500,602.61,1.174997e+07,604.9737,322826.0,"Meta Platforms, Inc. Class A Common Stock",1.535898e+12,XNAS,us
2844253,2026-05-20,META,600.7550,608.0000,597.8100,605.06,1.132971e+07,603.7783,309903.0,"Meta Platforms, Inc. Class A Common Stock",1.535898e+12,XNAS,us


# Merging

In [17]:
# Merge old FB with new META
df_metafb_fix = pd.concat([all_safe_df, fb_df], axis=0)

df_metafb_fix['timestamp'] = pd.to_datetime(df_metafb_fix['timestamp']).dt.date

#df_metafb_fix = super_combined_df.copy()

df_metafb_fix["timestamp"] = pd.to_datetime(df_metafb_fix["timestamp"])

# Keep FB through 2022-06-08
df_metafb_fix = df_metafb_fix[
    ~((df_metafb_fix["ticker"] == "FB") & (df_metafb_fix["timestamp"] >= "2022-06-09"))
]

# Keep META from 2022-06-09 onward
df_metafb_fix = df_metafb_fix[
    ~((df_metafb_fix["ticker"] == "META") & (df_metafb_fix["timestamp"] < "2022-06-09"))
]


# Change market cap value (old FB rows are null)
meta_market_cap = df_metafb_fix.loc[
    df_metafb_fix["ticker"] == "META",
    "market_cap"
].dropna().mean()

df_metafb_fix.loc[
    (df_metafb_fix["ticker"] == "FB") &
    (df_metafb_fix["market_cap"].isna()),
    "market_cap"
] = meta_market_cap

# Copy other metadata to ensure a match
df_metafb_fix.loc[df_metafb_fix["ticker"] == "FB", "name"] = "Meta Platforms, Inc. Class A Common Stock"
df_metafb_fix.loc[df_metafb_fix["ticker"] == "FB", "exchange"] = "XNAS"
df_metafb_fix.loc[df_metafb_fix["ticker"] == "FB", "country"] = "us"



# Replace old instances of FB with META
df_metafb_fix["ticker"] = df_metafb_fix["ticker"].replace({"FB": "META"})


# Sort
df_metafb_fix = df_metafb_fix.sort_values(by=['ticker', 'timestamp'])


There should now be no FB data, and META should stretch back to the beginning of the dataset.

In [18]:
df_metafb_fix[df_metafb_fix['ticker'] == 'FB']

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,name,market_cap,exchange,country


In [19]:
df_metafb_fix[df_metafb_fix['ticker'] == 'META']

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,name,market_cap,exchange,country
0,2021-05-24,META,318.210,325.9500,318.0300,324.63,1.644536e+07,324.1313,215147.0,"Meta Platforms, Inc. Class A Common Stock",1.535898e+12,XNAS,us
1,2021-05-25,META,327.080,329.1800,324.8000,327.79,1.638696e+07,327.5134,218086.0,"Meta Platforms, Inc. Class A Common Stock",1.535898e+12,XNAS,us
2,2021-05-26,META,328.350,329.8299,325.8200,327.66,9.686917e+06,327.5545,165262.0,"Meta Platforms, Inc. Class A Common Stock",1.535898e+12,XNAS,us
3,2021-05-27,META,328.000,333.7800,326.7600,332.75,2.047773e+07,331.3788,217767.0,"Meta Platforms, Inc. Class A Common Stock",1.535898e+12,XNAS,us
4,2021-05-28,META,331.000,332.8684,328.3300,328.73,1.200743e+07,330.2292,174406.0,"Meta Platforms, Inc. Class A Common Stock",1.535898e+12,XNAS,us
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2844250,2026-05-15,META,613.990,621.1999,609.3100,614.23,1.327257e+07,614.6824,309867.0,"Meta Platforms, Inc. Class A Common Stock",1.535898e+12,XNAS,us
2844251,2026-05-18,META,609.105,615.5900,603.6900,611.21,1.377244e+07,609.9810,376501.0,"Meta Platforms, Inc. Class A Common Stock",1.535898e+12,XNAS,us
2844252,2026-05-19,META,608.920,613.9300,600.5500,602.61,1.174997e+07,604.9737,322826.0,"Meta Platforms, Inc. Class A Common Stock",1.535898e+12,XNAS,us
2844253,2026-05-20,META,600.755,608.0000,597.8100,605.06,1.132971e+07,603.7783,309903.0,"Meta Platforms, Inc. Class A Common Stock",1.535898e+12,XNAS,us


I was having some duplicate problems. Going to diagnose here.

In [20]:
meta_df = df_metafb_fix[df_metafb_fix['ticker'] == 'META']
meta_df['name'].nunique()

1

# Further cleaning

I want to drop stocks which have missing days. dropna() won't work since those rows don't exist at all.

In [21]:
df_metafb_fix['ticker'].value_counts()

ticker
A       1255
METC    1255
MDWD    1255
MDXG    1255
MEC     1255
        ... 
AMSS       2
QADR       2
IPFX       2
LCLN       2
UCFI       1
Name: count, Length: 5041, dtype: int64

1255 is the number of days we will require.

In [22]:
import pandas as pd

ticker_counts = df_metafb_fix.set_index('timestamp')['ticker'].value_counts()

mask = df_metafb_fix['ticker'].isin(ticker_counts[ticker_counts >= 1255].index)

df_metafb_fix = df_metafb_fix[mask]

df_metafb_fix

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,name,market_cap,exchange,country
0,2021-05-24,A,133.51,134.41,132.5300,133.34,1.325305e+06,133.6936,18683.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us
1,2021-05-25,A,133.41,134.80,133.0100,133.23,1.905708e+06,133.6992,26967.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us
2,2021-05-26,A,136.30,138.00,133.2505,133.29,2.530998e+06,134.5671,31259.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us
3,2021-05-27,A,133.32,138.14,133.1600,137.54,3.720923e+06,137.2590,37177.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us
4,2021-05-28,A,138.60,139.21,138.0001,138.13,1.289049e+06,138.4188,20108.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4977185,2026-05-15,ZYME,24.50,25.34,23.6346,23.93,6.407959e+05,23.8908,11366.0,Zymeworks Inc.,1.827577e+09,XNAS,us
4977186,2026-05-18,ZYME,23.93,24.70,23.8800,24.43,7.502414e+05,24.3372,14709.0,Zymeworks Inc.,1.827577e+09,XNAS,us
4977187,2026-05-19,ZYME,24.19,24.55,23.8700,24.40,6.153201e+05,24.3494,12503.0,Zymeworks Inc.,1.827577e+09,XNAS,us
4977188,2026-05-20,ZYME,24.40,25.20,24.2650,24.84,5.426638e+05,24.8734,13051.0,Zymeworks Inc.,1.827577e+09,XNAS,us


Now we can drop nulls.

In [23]:
df_full_cleaned = df_metafb_fix.dropna()
df_full_cleaned

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,name,market_cap,exchange,country
0,2021-05-24,A,133.51,134.41,132.5300,133.34,1.325305e+06,133.6936,18683.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us
1,2021-05-25,A,133.41,134.80,133.0100,133.23,1.905708e+06,133.6992,26967.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us
2,2021-05-26,A,136.30,138.00,133.2505,133.29,2.530998e+06,134.5671,31259.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us
3,2021-05-27,A,133.32,138.14,133.1600,137.54,3.720923e+06,137.2590,37177.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us
4,2021-05-28,A,138.60,139.21,138.0001,138.13,1.289049e+06,138.4188,20108.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4977185,2026-05-15,ZYME,24.50,25.34,23.6346,23.93,6.407959e+05,23.8908,11366.0,Zymeworks Inc.,1.827577e+09,XNAS,us
4977186,2026-05-18,ZYME,23.93,24.70,23.8800,24.43,7.502414e+05,24.3372,14709.0,Zymeworks Inc.,1.827577e+09,XNAS,us
4977187,2026-05-19,ZYME,24.19,24.55,23.8700,24.40,6.153201e+05,24.3494,12503.0,Zymeworks Inc.,1.827577e+09,XNAS,us
4977188,2026-05-20,ZYME,24.40,25.20,24.2650,24.84,5.426638e+05,24.8734,13051.0,Zymeworks Inc.,1.827577e+09,XNAS,us


In [24]:
df_full_cleaned.to_csv('daily_data/daily_cleaned_1.csv', index=False)

# Everything below is likely redundant since I have now built a system to transform data as needed within a notebook, rather than wasting storage space on many variations of the same dataset with large redundancy. However, I will leave this code here as a reference in case my strategy needs to change in the future.

# SMAs, next day, previous day

In [25]:
import pandas as pd
df_full_cleaned = pd.read_csv('daily_data/daily_cleaned_1.csv')
df_full_cleaned

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,name,market_cap,exchange,country
0,2021-05-24,A,133.51,134.41,132.5300,133.34,1.325305e+06,133.6936,18683.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us
1,2021-05-25,A,133.41,134.80,133.0100,133.23,1.905708e+06,133.6992,26967.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us
2,2021-05-26,A,136.30,138.00,133.2505,133.29,2.530998e+06,134.5671,31259.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us
3,2021-05-27,A,133.32,138.14,133.1600,137.54,3.720923e+06,137.2590,37177.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us
4,2021-05-28,A,138.60,139.21,138.0001,138.13,1.289049e+06,138.4188,20108.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3493827,2026-05-15,ZYME,24.50,25.34,23.6346,23.93,6.407959e+05,23.8908,11366.0,Zymeworks Inc.,1.827577e+09,XNAS,us
3493828,2026-05-18,ZYME,23.93,24.70,23.8800,24.43,7.502414e+05,24.3372,14709.0,Zymeworks Inc.,1.827577e+09,XNAS,us
3493829,2026-05-19,ZYME,24.19,24.55,23.8700,24.40,6.153201e+05,24.3494,12503.0,Zymeworks Inc.,1.827577e+09,XNAS,us
3493830,2026-05-20,ZYME,24.40,25.20,24.2650,24.84,5.426638e+05,24.8734,13051.0,Zymeworks Inc.,1.827577e+09,XNAS,us


In [26]:
df_full_cleaned['SMA10'] = df_full_cleaned.groupby('ticker')['close'].transform(lambda x: x.rolling(10).mean())
df_full_cleaned['SMA20'] = df_full_cleaned.groupby('ticker')['close'].transform(lambda x: x.rolling(20).mean())
df_full_cleaned['SMA50'] = df_full_cleaned.groupby('ticker')['close'].transform(lambda x: x.rolling(50).mean())
df_full_cleaned['SMA100'] = df_full_cleaned.groupby('ticker')['close'].transform(lambda x: x.rolling(100).mean())

In [27]:
df_full_cleaned['pct_change_from_prev'] = (
    df_full_cleaned.groupby('ticker')['close']
    .pct_change() * 100
)

df_full_cleaned['next_close_pct_change'] = (
    (
        df_full_cleaned.groupby('ticker')['close']
        .shift(-1)
        / df_full_cleaned['close']
    - 1
    ) * 100
)

df_full_cleaned

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,name,market_cap,exchange,country,SMA10,SMA20,SMA50,SMA100,pct_change_from_prev,next_close_pct_change
0,2021-05-24,A,133.51,134.41,132.5300,133.34,1.325305e+06,133.6936,18683.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us,NaN,NaN,NaN,NaN,NaN,-0.082496
1,2021-05-25,A,133.41,134.80,133.0100,133.23,1.905708e+06,133.6992,26967.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us,NaN,NaN,NaN,NaN,-0.082496,0.045035
2,2021-05-26,A,136.30,138.00,133.2505,133.29,2.530998e+06,134.5671,31259.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us,NaN,NaN,NaN,NaN,0.045035,3.188536
3,2021-05-27,A,133.32,138.14,133.1600,137.54,3.720923e+06,137.2590,37177.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us,NaN,NaN,NaN,NaN,3.188536,0.428966
4,2021-05-28,A,138.60,139.21,138.0001,138.13,1.289049e+06,138.4188,20108.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us,NaN,NaN,NaN,NaN,0.428966,-0.673279
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3493827,2026-05-15,ZYME,24.50,25.34,23.6346,23.93,6.407959e+05,23.8908,11366.0,Zymeworks Inc.,1.827577e+09,XNAS,us,25.195,26.5305,25.6757,24.81475,-0.125209,2.089427
3493828,2026-05-18,ZYME,23.93,24.70,23.8800,24.43,7.502414e+05,24.3372,14709.0,Zymeworks Inc.,1.827577e+09,XNAS,us,24.960,26.3595,25.6827,24.78285,2.089427,-0.122800
3493829,2026-05-19,ZYME,24.19,24.55,23.8700,24.40,6.153201e+05,24.3494,12503.0,Zymeworks Inc.,1.827577e+09,XNAS,us,24.732,26.1660,25.6857,24.75895,-0.122800,1.803279
3493830,2026-05-20,ZYME,24.40,25.20,24.2650,24.84,5.426638e+05,24.8734,13051.0,Zymeworks Inc.,1.827577e+09,XNAS,us,24.455,25.9815,25.6733,24.73875,1.803279,1.086957


In [28]:
super_safe_df = df_full_cleaned.copy()


In [29]:
df = super_safe_df

# CALLING FROM FILE
with open("daily_data/COMMAND calculate pct of first day.txt", "r") as file:
    code = file.read()
    exec(code)

In [30]:
super_safe_df

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,name,market_cap,exchange,country,SMA10,SMA20,SMA50,SMA100,pct_change_from_prev,next_close_pct_change,close_pct_of_day1
0,2021-05-24,A,133.51,134.41,132.5300,133.34,1.325305e+06,133.6936,18683.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us,NaN,NaN,NaN,NaN,NaN,-0.082496,100.000000
1,2021-05-25,A,133.41,134.80,133.0100,133.23,1.905708e+06,133.6992,26967.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us,NaN,NaN,NaN,NaN,-0.082496,0.045035,99.917504
2,2021-05-26,A,136.30,138.00,133.2505,133.29,2.530998e+06,134.5671,31259.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us,NaN,NaN,NaN,NaN,0.045035,3.188536,99.962502
3,2021-05-27,A,133.32,138.14,133.1600,137.54,3.720923e+06,137.2590,37177.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us,NaN,NaN,NaN,NaN,3.188536,0.428966,103.149843
4,2021-05-28,A,138.60,139.21,138.0001,138.13,1.289049e+06,138.4188,20108.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us,NaN,NaN,NaN,NaN,0.428966,-0.673279,103.592320
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3493827,2026-05-15,ZYME,24.50,25.34,23.6346,23.93,6.407959e+05,23.8908,11366.0,Zymeworks Inc.,1.827577e+09,XNAS,us,25.195,26.5305,25.6757,24.81475,-0.125209,2.089427,82.975035
3493828,2026-05-18,ZYME,23.93,24.70,23.8800,24.43,7.502414e+05,24.3372,14709.0,Zymeworks Inc.,1.827577e+09,XNAS,us,24.960,26.3595,25.6827,24.78285,2.089427,-0.122800,84.708738
3493829,2026-05-19,ZYME,24.19,24.55,23.8700,24.40,6.153201e+05,24.3494,12503.0,Zymeworks Inc.,1.827577e+09,XNAS,us,24.732,26.1660,25.6857,24.75895,-0.122800,1.803279,84.604716
3493830,2026-05-20,ZYME,24.40,25.20,24.2650,24.84,5.426638e+05,24.8734,13051.0,Zymeworks Inc.,1.827577e+09,XNAS,us,24.455,25.9815,25.6733,24.73875,1.803279,1.086957,86.130374


In [75]:
# MANUAL VERSION WORKED
super_safe_df['close_pct_of_day1'] = (super_safe_df.groupby('ticker')['close'].transform(lambda x: (x / x.iloc[0]) * 100))
super_safe_df

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,name,market_cap,exchange,country,SMA10,SMA20,SMA50,SMA100,pct_change_from_prev,next_close_pct_change,close_pct_of_day1
0,2021-05-24,A,133.51,134.41,132.5300,133.34,1.325305e+06,133.6936,18683.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us,NaN,NaN,NaN,NaN,NaN,-0.082496,100.000000
1,2021-05-25,A,133.41,134.80,133.0100,133.23,1.905708e+06,133.6992,26967.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us,NaN,NaN,NaN,NaN,-0.082496,0.045035,99.917504
2,2021-05-26,A,136.30,138.00,133.2505,133.29,2.530998e+06,134.5671,31259.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us,NaN,NaN,NaN,NaN,0.045035,3.188536,99.962502
3,2021-05-27,A,133.32,138.14,133.1600,137.54,3.720923e+06,137.2590,37177.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us,NaN,NaN,NaN,NaN,3.188536,0.428966,103.149843
4,2021-05-28,A,138.60,139.21,138.0001,138.13,1.289049e+06,138.4188,20108.0,Agilent Technologies Inc.,3.215449e+10,XNYS,us,NaN,NaN,NaN,NaN,0.428966,-0.673279,103.592320
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3493827,2026-05-15,ZYME,24.50,25.34,23.6346,23.93,6.407959e+05,23.8908,11366.0,Zymeworks Inc.,1.827577e+09,XNAS,us,25.195,26.5305,25.6757,24.81475,-0.125209,2.089427,82.975035
3493828,2026-05-18,ZYME,23.93,24.70,23.8800,24.43,7.502414e+05,24.3372,14709.0,Zymeworks Inc.,1.827577e+09,XNAS,us,24.960,26.3595,25.6827,24.78285,2.089427,-0.122800,84.708738
3493829,2026-05-19,ZYME,24.19,24.55,23.8700,24.40,6.153201e+05,24.3494,12503.0,Zymeworks Inc.,1.827577e+09,XNAS,us,24.732,26.1660,25.6857,24.75895,-0.122800,1.803279,84.604716
3493830,2026-05-20,ZYME,24.40,25.20,24.2650,24.84,5.426638e+05,24.8734,13051.0,Zymeworks Inc.,1.827577e+09,XNAS,us,24.455,25.9815,25.6733,24.73875,1.803279,1.086957,86.130374


# These below now likely redundant after creating command system to transform data.

# Save with nulls

In [31]:
df_full_cleaned.to_csv('daily_data/daily_cleaned_2a.csv', index=False)

# Drop nulls and save without

In [32]:
df_full_cleaned = df_full_cleaned.dropna()
df_full_cleaned.to_csv('daily_data/daily_cleaned_2b.csv', index=False)